# Alpha Research — Homework 2
**Luca Ragazzi & Jaskaran Kalra**

### Objective
Construction of a point-in-time S&P 500 universe, collection of market and alternative data, data quality analysis, cross-referencing, and a first OLS modelling attempt.

### Sample Period
January 2010 — May 2026

### Universe
S&P 500 constituents as of January 1, 2010, sourced from the fja05680/sp500 GitHub repository — a point-in-time source tracking all historical additions and removals since 1996.

In [90]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime
import os
import urllib.request
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# ── Create data directory ─────────────────────────────────────────────
os.makedirs('data', exist_ok=True)

In [91]:
import time

In [92]:
# ── Load historical S&P 500 constituents ─────────────────────────────
pit = pd.read_csv('../data/sp500/S&P 500 Historical Components & Changes(01-17-2026).csv')
pit['date'] = pd.to_datetime(pit['date'])

# ── Extract PIT snapshot at 2010-01-01 ───────────────────────────────
pit_date = pd.Timestamp('2010-01-01')
snapshot = pit[pit['date'] <= pit_date].iloc[-1]

print(f"Snapshot date: {snapshot['date'].date()}")

# ── Parse tickers ─────────────────────────────────────────────────────
tickers = [t.strip() for t in snapshot['tickers'].split(',')]

print(f"Constituents at {pit_date.date()}: {len(tickers)}")
print(f"First 10 tickers: {tickers[:10]}")

Snapshot date: 2009-12-30
Constituents at 2010-01-01: 499
First 10 tickers: ['A', 'AABA', 'AAPL', 'ABC', 'ABT', 'ACS', 'ADBE', 'ADI', 'ADM', 'ADP']


In [93]:
# ── Download prices from yfinance ────────────────────────────────────
prices = yf.download(
    tickers,
    start='2010-01-01',
    end='2026-05-16',
    auto_adjust=True
)['Close']

print(f"Downloaded prices shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")

$TWC: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-05-16)
[                       0%                       ]$MWW: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-05-16)
$LO: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-05-16)
[                       0%                       ]  2 of 499 completed$HNZ: possibly delisted; no price data found  (1d 2010-01-01 -> 2026-05-16)
$FII: possibly delisted; no timezone found
[                       1%                       ]  4 of 499 completed$NBL: possibly delisted; no timezone found
[                       1%                       ]  5 of 499 completed$COG: possibly delisted; no timezone found
[*                      2%                       ]  11 of 499 completed$RRD: possibly delisted; no timezone found
[*                      3%                       ]  14 of 499 completed$AABA: possibly delisted; no timezone found
$MRO: possibly delisted; no timezone found
[**                     4%          

Downloaded prices shape: (4117, 499)
Date range: 2010-01-04 to 2026-05-15


## Survivorship Bias Assessment

Despite using a true point-in-time constituent list, 130 tickers 
(26% of the universe) could not be downloaded from yfinance due to 
delisting or data unavailability. This reintroduces survivorship bias 
comparable to using a current constituent list. Eliminating this bias 
entirely would require a commercial database such as CRSP or Compustat.

In [94]:
# ── Document and handle failed downloads ─────────────────────────────
failed_tickers = [
    'AABA', 'ABC', 'ACS', 'AGN', 'AKS', 'ALTR', 'ANTM', 'APOL', 'ARG', 'ARNC',
    'ATGE', 'AVP', 'AYE', 'BCR', 'BDK', 'BF.B', 'BHGE', 'BIG', 'BJS', 'BLL',
    'BNI', 'BRCM', 'BTUUQ', 'CBS', 'CELG', 'CEPH', 'CFN', 'CHK', 'CMA', 'COG',
    'CTL', 'CTXS', 'CVH', 'DF', 'DFS', 'DNB', 'DNR', 'DO', 'DTV', 'EKDKQ',
    'ETFC', 'FDO', 'FII', 'FLIR', 'FRX', 'FTR', 'GAS', 'GPS', 'HCBK', 'HCP',
    'HES', 'HNZ', 'HRS', 'HSH', 'HSP', 'IGT', 'IPG', 'JCP', 'JEC', 'JNPR',
    'JNS', 'JWN', 'K', 'LLL', 'LLTC', 'LM', 'LO', 'LSI', 'LXK', 'MDP',
    'MFE', 'MIL', 'MJN', 'MMC', 'MON', 'MRO', 'MWV', 'MWW', 'MYL', 'NBL',
    'NOVL', 'NVLS', 'NYX', 'ODP', 'PBCT', 'PCP', 'PDCO', 'PGN', 'PKI', 'PLL',
    'PX', 'PXD', 'QLGC', 'RAI', 'RDC', 'RHT', 'RRD', 'RSHCQ', 'RTN', 'SIAL',
    'SNI', 'SRCL', 'STJ', 'STR', 'SUNEQ', 'SWN', 'SWY', 'SYMC', 'TGNA', 'TIF',
    'TMK', 'TSS', 'TWC', 'UTX', 'VAR', 'VIAB', 'WBA', 'WFM', 'WIN', 'WYND',
    'X', 'XL', 'XLNX', 'XTO',
]

# Note: 124 tickers failed to download. These are companies that were
# delisted, acquired, or merged during the 2010-2026 period.
# For companies with partial data, yfinance returns prices up to their
# last trading day — those observations remain in the dataset.
# Only companies with zero data are excluded here.

successful_tickers = [t for t in tickers if t not in failed_tickers]
prices = prices[successful_tickers]

print(f"Successful downloads : {len(successful_tickers)}")
print(f"Failed downloads     : {len(failed_tickers)}")
print(f"Final prices shape   : {prices.shape}")

# ── Save raw prices ───────────────────────────────────────────────────
prices.to_csv('data/sp500_prices_raw.csv')
print("Prices saved to data/sp500_prices_raw.csv")

Successful downloads : 375
Failed downloads     : 124
Final prices shape   : (4117, 375)
Prices saved to data/sp500_prices_raw.csv


## 3. Data Quality Analysis — Prices

In [95]:
# ── Data Quality Analysis — Prices ───────────────────────────────────
print(f"Shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Tickers: {len(prices.columns)}")

# Missing values per ticker
missing_pct = prices.isnull().sum() / len(prices) * 100
print(f"\nMissing values per ticker (%):")
print(missing_pct.describe())

# Tickers with more than 20% missing
high_missing = missing_pct[missing_pct > 20]
print(f"\nTickers with >20% missing values: {len(high_missing)}")
print(high_missing.sort_values(ascending=False).head(10))

Shape: (4117, 375)
Date range: 2010-01-04 to 2026-05-15
Tickers: 375

Missing values per ticker (%):
count    375.000000
mean       8.403789
std       23.471335
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       98.712655
dtype: float64

Tickers with >20% missing values: 46
Ticker
RX      98.712655
APC     98.421180
LIFE    98.178285
SPLS    97.983969
CCE     96.648045
Q       96.623755
POM     96.307991
CAM     96.259412
PCL     95.214962
SNDK    92.348798
dtype: float64


### Missing Values Treatment

Tickers with more than 50% missing values are removed — these companies 
were delisted too early in the sample period to provide meaningful data. 
Tickers with partial data (delisted mid-sample) are retained as their 
valid observations contribute to reducing survivorship bias.

In [96]:
# ── Remove tickers with excessive missing values ───────────────────────
# Threshold: remove tickers with more than 50% missing values
# Tickers with <50% missing have sufficient data for meaningful analysis
threshold = 50
missing_pct = prices.isnull().sum() / len(prices) * 100

tickers_to_keep = missing_pct[missing_pct <= threshold].index.tolist()
prices_clean = prices[tickers_to_keep]

print(f"Prices shape before: {prices.shape}")
print(f"Prices shape after:  {prices_clean.shape}")
print(f"Tickers removed (>{threshold}% missing): {prices.shape[1] - prices_clean.shape[1]}")

# Save
prices_clean.to_csv('data/sp500_prices_clean.csv')
print(f"\nClean prices saved — {prices_clean.shape[1]} tickers retained")

Prices shape before: (4117, 375)
Prices shape after:  (4117, 341)
Tickers removed (>50% missing): 34

Clean prices saved — 341 tickers retained


## 4. Predictor Datasets

Four datasets are downloaded as potential predictors of future returns:
1. **VIX** — market fear index (sentiment)
2. **Fama-French Factors** — Mkt-RF, SMB, HML (asset pricing)
3. **FRED** — Fed Funds Rate, CPI, Unemployment Rate (macroeconomic)
4. **Momentum** — 12-month return excluding last month (technical)

In [97]:
# ── 1. VIX ───────────────────────────────────────────────────────────
vix = yf.download('^VIX', start='2010-01-01', end='2026-05-16', auto_adjust=True)['Close']
vix.to_csv('data/vix.csv')
print(f"VIX shape: {vix.shape}")

# ── 2. Fama-French ───────────────────────────────────────────────────
ff_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip'
ff = pd.read_csv(ff_url, skiprows=3, index_col=0)
ff = ff[ff.index.astype(str).str.match(r'^\d{8}$')]
ff.index = pd.to_datetime(ff.index, format='%Y%m%d')
ff = ff.loc['2010-01-01':'2026-05-16']
ff.to_csv('data/famafrench.csv')
print(f"Fama-French shape: {ff.shape}")

# ── 3. FRED ──────────────────────────────────────────────────────────
FRED_API_KEY = 'fab4104d00c3061dee2126deb258f448'

def get_fred(series_id):
    url = f'https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json'
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read())
    df = pd.DataFrame(data['observations'])[['date', 'value']]
    df.columns = ['date', series_id]
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    df[series_id] = pd.to_numeric(df[series_id], errors='coerce')
    return df

# Download from 2009 to have 12 months history for CPI YoY calculation
fed_funds    = get_fred('FEDFUNDS')
cpi          = get_fred('CPIAUCSL')
time.sleep(1)
unemployment = get_fred('UNRATE')
fred = fed_funds.join(cpi).join(unemployment)
fred = fred.loc['2009-01-01':'2026-05-16']
fred.to_csv('data/fred.csv')
print(f"FRED shape: {fred.shape}")

# ── 4. Momentum ───────────────────────────────────────────────────────
# 12-month return excluding last month (Jegadeesh & Titman 1993)
momentum = np.log(prices_clean).diff(252).shift(21)
momentum.to_csv('data/momentum.csv')
print(f"Momentum shape: {momentum.shape}")

print(f"\nAll datasets saved to data")

[*********************100%***********************]  1 of 1 completed


VIX shape: (4117, 1)
Fama-French shape: (4106, 4)
FRED shape: (209, 3)
Momentum shape: (4117, 341)

All datasets saved to data


## 5. Data Quality Analysis — Predictor Datasets

In [98]:
# ── Data Quality — Predictor Datasets ────────────────────────────────
datasets = {'VIX': vix, 'Fama-French': ff, 'FRED': fred}

for name, df in datasets.items():
    print(f"{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Shape:          {df.shape}")
    print(f"Date range:     {df.index[0].date()} to {df.index[-1].date()}")
    print(f"Missing values: {df.isnull().sum().sum()}")
    print(f"Descriptive statistics:")
    print(df.describe())
    print()

VIX
Shape:          (4117, 1)
Date range:     2010-01-04 to 2026-05-15
Missing values: 0
Descriptive statistics:
Ticker         ^VIX
count   4117.000000
mean      18.448703
std        6.824620
min        9.140000
25%       13.810000
50%       16.770000
75%       21.129999
max       82.690002

Fama-French
Shape:          (4106, 4)
Date range:     2010-01-04 to 2026-04-30
Missing values: 0
Descriptive statistics:
            Mkt-RF          SMB          HML           RF
count  4106.000000  4106.000000  4106.000000  4106.000000
mean      0.053371    -0.002815    -0.002808     0.005599
std       1.118207     0.618885     0.777043     0.007955
min     -12.010000    -3.530000    -5.030000     0.000000
25%      -0.400000    -0.380000    -0.380000     0.000000
50%       0.080000    -0.010000    -0.025000     0.000000
75%       0.597500     0.360000     0.360000     0.010000
max       9.650000     5.450000     6.730000     0.020000

FRED
Shape:          (209, 3)
Date range:     2009-01-01 to 20

In [99]:
# ── Check FRED missing values ─────────────────────────────────────────
print("FRED missing values:")
print(fred[fred.isnull().any(axis=1)])

FRED missing values:
            FEDFUNDS  CPIAUCSL  UNRATE
date                                  
2025-10-01      4.09       NaN     NaN
2026-05-01      3.63       NaN     NaN


In [100]:
# ── Forward fill FRED missing values ─────────────────────────────────
# October 2025: CPI and Unemployment not yet published at download time
# Forward fill with September 2025 values — conservative assumption
fred = fred.ffill()
print(f"Missing values after ffill: {fred.isnull().sum().sum()}")

Missing values after ffill: 0


### FRED Missing Values

One observation (2025-10-01) contained missing values for CPI and Unemployment Rate,
likely due to publication lag at the time of download. Forward-filled with the 
most recent available values — a conservative and point-in-time consistent approach.

In [101]:
# ── Outlier Analysis — Daily Returns ─────────────────────────────────
returns = prices_clean.pct_change().dropna()

print("Daily Returns — Descriptive Statistics:")
print(returns.stack().describe())

print(f"\nExtreme returns (> 50% or < -50% in a single day):")
extreme_mask = (returns > 0.5) | (returns < -0.5)

for ticker in returns.columns:
    ticker_extreme = returns[ticker][extreme_mask[ticker]].dropna()
    if len(ticker_extreme) > 0:
        for date, value in ticker_extreme.items():
            print(f"  {ticker:6s} | {date.date()} | {value*100:+.1f}%")

Daily Returns — Descriptive Statistics:
count    43989.000000
mean         0.000840
std          0.022973
min         -0.866667
25%         -0.006699
50%          0.000849
75%          0.008496
max          1.833945
dtype: float64

Extreme returns (> 50% or < -50% in a single day):
  CPWR   | 2017-11-20 | -86.7%
  CPWR   | 2017-11-22 | +76.4%
  CPWR   | 2017-11-24 | +183.4%
  EP     | 2018-01-09 | +73.1%
  EP     | 2018-04-27 | +53.8%
  EP     | 2018-05-17 | +92.3%
  EP     | 2018-05-22 | -56.0%
  EP     | 2018-05-24 | +127.3%
  EP     | 2018-06-05 | -56.0%


In [103]:
# ── Outlier Analysis — 21-day Returns ────────────────────────────────
# Daily return screening is insufficient — corrupted tickers can drift
# gradually without triggering a single-day ±50% threshold.
# Note: EP, CPWR, TIE, BMC have already been removed by the daily screen above.
# This screen catches any remaining tickers with extreme 21-day returns.

returns_21d = prices_clean.pct_change(21)

print("Tickers with any 21-day return > 500%:")
extreme_21d = (returns_21d > 5).any()
extreme_tickers = extreme_21d[extreme_21d].index.tolist()
print(extreme_tickers)

print("\nMax 21-day return per flagged ticker:")
for t in extreme_tickers:
    print(f"  {t}: {returns_21d[t].max()*100:.1f}%")

Tickers with any 21-day return > 500%:
['BMC', 'CPWR', 'EP', 'GME', 'MI', 'NBR', 'TIE']

Max 21-day return per flagged ticker:
  BMC: 3030.1%
  CPWR: 5100.0%
  EP: 1020.0%
  GME: 1624.6%
  MI: 1530.3%
  NBR: 528.1%
  TIE: 992757.2%


In [104]:
# ── Remove corrupted tickers ──────────────────────────────────────────
# Daily screen: EP, CPWR — corrupted due to ticker reassignment after delisting
# 21-day screen: TIE, BMC — corrupted due to ticker reassignment
# 21-day screen: GME, MI, NBR — real events but extreme outliers (+500% to +1600%)
#                that are practically untradeable and add noise not signal
corrupted = ['EP', 'CPWR', 'TIE', 'BMC', 'GME', 'MI', 'NBR']
prices_clean = prices_clean.drop(columns=corrupted, errors='ignore')
print(f"Removed tickers: {corrupted}")
print(f"Final prices shape: {prices_clean.shape}")

prices_clean.to_csv('data/sp500_prices_clean.csv')
print("Updated clean prices saved")

Removed tickers: ['EP', 'CPWR', 'TIE', 'BMC', 'GME', 'MI', 'NBR']
Final prices shape: (4117, 334)
Updated clean prices saved


### Outlier Assessment

**Daily return screen (±50% threshold):**

Two tickers (EP, CPWR) exhibit clearly corrupted price data due to ticker reassignment after delisting — alternating extreme single-day returns with no plausible economic explanation. Both are removed.

**21-day return screen (>500% threshold):**

The daily screen is insufficient — corrupted tickers can drift gradually without triggering a single-day threshold. Five additional tickers are flagged:

- **TIE** (Titanium Metals, acquired 2012): ticker reuse produces 21-day returns of +992,757% — clearly corrupted data.
- **BMC** (BMC Software, delisted 2013): ticker reuse produces 21-day returns exceeding +3,000%.
- **GME** (GameStop, January 2021): real short squeeze event but 21-day returns of +1,625% are practically untradeable and unforecastable — removed to reduce noise.
- **MI, NBR**: volatile small caps with 21-day returns exceeding +500% — extreme outliers that distort the model without adding predictive signal.

All seven tickers are removed from the dataset.

## 6. Cross-Reference and Panel Construction

All datasets are merged into a single long-format panel where each row represents 
one ticker × date observation. Key challenges addressed:
- **Frequency mismatch**: FRED is monthly, all others daily → forward-filled to daily
- **Granularity mismatch**: VIX, Fama-French, FRED are market-wide; prices and momentum are ticker-specific
- **CPI transformation**: year-on-year percentage change to ensure stationarity

### FRED Publication Lag — Point-in-Time Correction

FRED dates monthly observations at the **start of the reference period**: the January 2025 value carries index `2025-01-01`. In reality the data is published days later:

| Series | Typical release | Shift applied |
|---|---|---|
| `FEDFUNDS` | ~5 business days after month end | +5 days |
| `UNRATE` | First Friday of following month (~7 days) | +7 days |
| `CPIAUCSL` | ~13 days after month end | +13 days |

Each series is shifted independently by its actual publication lag before resampling to daily frequency. This avoids look-ahead bias while minimising the unnecessary data loss of a blanket one-month shift.

In [105]:
# ── CPI Year-on-Year transformation ──────────────────────────────────
if 'CPIAUCSL' in fred.columns:
    fred['CPI_YoY'] = fred['CPIAUCSL'].pct_change(12) * 100
    fred = fred.drop(columns=['CPIAUCSL'])

# ── Point-in-time correction: series-specific publication lags ────────
# Each series is shifted and resampled independently to avoid index misalignment
# FEDFUNDS: H.15 release ~5 days after month end
# UNRATE:   Employment Situation, first Friday of following month (~7 days)
# CPI_YoY:  CPI released ~13 days after month end

def shift_and_resample(series, days, daily_index):
    shifted = pd.Series(series.values, index=series.index + pd.DateOffset(days=days))
    daily = shifted.resample('D').ffill()
    return daily.reindex(daily_index, method='ffill')

fred_daily = pd.DataFrame({
    'FEDFUNDS': shift_and_resample(fred['FEDFUNDS'],  5, prices_clean.index),
    'UNRATE':   shift_and_resample(fred['UNRATE'],    7, prices_clean.index),
    'CPI_YoY':  shift_and_resample(fred['CPI_YoY'], 13, prices_clean.index),
})

# ── Align all datasets to prices index ───────────────────────────────
vix_aligned      = vix.reindex(prices_clean.index, method='ffill')
ff_aligned       = ff.reindex(prices_clean.index, method='ffill')
fred_aligned     = fred_daily
momentum_aligned = momentum.reindex(prices_clean.index)

# ── Compute forward simple returns ─────────────────────────────
forward_returns = prices_clean.pct_change(21).shift(-21)

print(f"All datasets aligned to {len(prices_clean.index)} trading days")
print(f"Date range: {prices_clean.index[0].date()} to {prices_clean.index[-1].date()}")
print(f"\nFRED missing values after lag correction:\n{fred_daily.isnull().sum()}")

# ── Build long-format panel ───────────────────────────────────────────
panels = []

for ticker in prices_clean.columns:
    df = pd.DataFrame({
        'date'           : prices_clean.index,
        'ticker'         : ticker,
        'forward_return' : forward_returns[ticker].values,
        'momentum'       : momentum_aligned[ticker].values,
        'VIX'            : vix_aligned.values.flatten(),
        'Mkt_RF'         : ff_aligned['Mkt-RF'].values,
        'SMB'            : ff_aligned['SMB'].values,
        'HML'            : ff_aligned['HML'].values,
        'FEDFUNDS'       : fred_aligned['FEDFUNDS'].values,
        'CPI_YoY'        : fred_aligned['CPI_YoY'].values,
        'UNRATE'         : fred_aligned['UNRATE'].values,
    })
    panels.append(df)

panel = pd.concat(panels, ignore_index=True)

print(f"\nPanel shape: {panel.shape}")
print(f"Missing values:\n{panel.isnull().sum()}")

All datasets aligned to 4117 trading days
Date range: 2010-01-04 to 2026-05-15

FRED missing values after lag correction:
FEDFUNDS    0
UNRATE      0
CPI_YoY     8
dtype: int64

Panel shape: (1375078, 11)
Missing values:
date                   0
ticker                 0
forward_return     24516
momentum          108510
VIX                    0
Mkt_RF                 0
SMB                    0
HML                    0
FEDFUNDS               0
CPI_YoY             2672
UNRATE                 0
dtype: int64


In [106]:
panel

,date,ticker,forward_return,momentum,VIX,Mkt_RF,SMB,HML,FEDFUNDS,CPI_YoY,UNRATE
0,2010-01-04,A,-0.056230,NaN,20.040001,1.69,0.61,1.14,0.12,NaN,9.9
1,2010-01-05,A,-0.061047,NaN,19.350000,0.31,-0.64,1.22,0.12,NaN,9.9
2,2010-01-06,A,-0.054457,NaN,19.160000,0.13,-0.23,0.55,0.11,NaN,9.9
3,2010-01-07,A,-0.052256,NaN,19.059999,0.40,0.09,0.96,0.11,NaN,9.9
4,2010-01-08,A,-0.045130,NaN,18.129999,0.33,0.36,0.02,0.11,NaN,9.8
...,...,...,...,...,...,...,...,...,...,...,...
1375073,2026-05-11,ZION,NaN,0.402886,18.379999,1.13,0.98,-0.71,3.63,3.779246,4.3
1375074,2026-05-12,ZION,NaN,0.329672,17.990000,1.13,0.98,-0.71,3.63,3.779246,4.3
1375075,2026-05-13,ZION,NaN,0.406225,17.870001,1.13,0.98,-0.71,3.63,3.779246,4.3
1375076,2026-05-14,ZION,NaN,0.419944,17.260000,1.13,0.98,-0.71,3.63,3.676315,4.3


In [107]:
# ── Drop NaN rows ─────────────────────────────────────────────────────
panel_clean = panel.dropna(subset=['forward_return', 'momentum'])

print(f"Panel before dropna: {panel.shape}")
print(f"Panel after dropna:  {panel_clean.shape}")
print(f"Rows dropped:        {len(panel) - len(panel_clean)}")
print(f"\nMissing values after dropna:")
print(panel_clean.isnull().sum())

# ── Save panel ────────────────────────────────────────────────────────
panel_clean.to_csv('data/panel_clean.csv', index=False)
print(f"\nPanel saved to data/panel_clean.csv")

Panel before dropna: (1375078, 11)
Panel after dropna:  (1259364, 11)
Rows dropped:        115714

Missing values after dropna:
date              0
ticker            0
forward_return    0
momentum          0
VIX               0
Mkt_RF            0
SMB               0
HML               0
FEDFUNDS          0
CPI_YoY           0
UNRATE            0
dtype: int64

Panel saved to data/panel_clean.csv


### Panel Construction

The final panel contains **1,271,523 observations** across 338 tickers and 4,117 trading days. Rows with NaN in `forward_return` (last 21 days per ticker) and `momentum` (first 252 days per ticker) are dropped by construction — these observations cannot be used in the regression. The CPI_YoY NaNs from the 13-day publication lag fall entirely within the momentum warmup window and are removed in the same step.

## 7. Data Transformations

Before running OLS, the following transformations are applied to address violations 
of classical assumptions:

- **Forward return** → winsorized at 1st and 99th percentile (fat tails)
- **Momentum** → winsorized at 1st and 99th percentile (fat tails)  
- **All features** → standardized (z-score) for comparability of coefficients
- **Forward return** → log-return for symmetry and stationarity

In [108]:
from scipy.stats import mstats
# ── Step 1: Convert forward return to log-return ──────────────────────
# Log-returns are more symmetric and stationary than simple returns
panel_clean = panel_clean.copy()
panel_clean['forward_return'] = np.log(1 + panel_clean['forward_return'])

# ── Step 2: Winsorize forward return and momentum ─────────────────────
# Cap extreme values at 1st and 99th percentile to reduce outlier influence
for col in ['forward_return', 'momentum']:
    panel_clean[col] = mstats.winsorize(panel_clean[col], limits=[0.01, 0.01])

print("Winsorization applied to forward_return and momentum")
print(f"forward_return range: {panel_clean['forward_return'].min():.4f} to {panel_clean['forward_return'].max():.4f}")
print(f"momentum range:       {panel_clean['momentum'].min():.4f} to {panel_clean['momentum'].max():.4f}")

# ── Step 3: Standardize all features ─────────────────────────────────
# Z-score normalization for comparability of OLS coefficients
features = ['momentum', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

for col in features:
    mean = panel_clean[col].mean()
    std  = panel_clean[col].std()
    panel_clean[col] = (panel_clean[col] - mean) / std

print("\nStandardization applied to all features")
print(panel_clean[features].describe().round(4))

Winsorization applied to forward_return and momentum
forward_return range: -0.2512 to 0.2285
momentum range:       -0.7173 to 0.8064

Standardization applied to all features
           momentum           VIX        Mkt_RF           SMB           HML  \
count  1.259364e+06  1.259364e+06  1.259364e+06  1.259364e+06  1.259364e+06   
mean  -0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00 -0.000000e+00   
std    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   
min   -3.155800e+00 -1.316600e+00 -1.080350e+01 -5.683600e+00 -6.364600e+00   
25%   -5.376000e-01 -6.636000e-01 -4.041000e-01 -6.037000e-01 -4.901000e-01   
50%    5.680000e-02 -2.627000e-01  1.690000e-02 -7.000000e-03 -3.440000e-02   
75%    5.950000e-01  3.641000e-01  4.827000e-01  5.736000e-01  4.721000e-01   
max    2.705400e+00  9.404600e+00  8.598000e+00  8.798200e+00  8.524200e+00   

           FEDFUNDS       CPI_YoY        UNRATE  
count  1.259364e+06  1.259364e+06  1.259364e+06  
mean   0.00000

## 8. OLS Regression

In [109]:
import statsmodels.api as sm

# ── Prepare X and y ───────────────────────────────────────────────────
features = ['momentum', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

X = panel_clean[features]
y = panel_clean['forward_return']

# Add constant for intercept
X = sm.add_constant(X)

# ── Run OLS ───────────────────────────────────────────────────────────
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         forward_return   R-squared:                       0.025
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     4048.
Date:                Wed, 03 Jun 2026   Prob (F-statistic):               0.00
Time:                        13:53:41   Log-Likelihood:             1.4183e+06
No. Observations:             1259364   AIC:                        -2.837e+06
Df Residuals:                 1259355   BIC:                        -2.837e+06
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0086   6.99e-05    123.427      0.0

### OLS Results — First Attempt

The pooled OLS regression on the full daily panel yields an R² of **2.5%**, which is 
realistic for financial return prediction. All coefficients are statistically significant 
except **HML** (p=0.181) — the value factor does not contribute meaningfully in this sample.
The economic magnitude of all coefficients is negligible, a direct consequence of having 
1.26 million observations where even trivial effects become significant.

Two diagnostic tests reveal serious violations of OLS assumptions:

- **Durbin-Watson = 0.109**: indicates near-perfect positive serial autocorrelation in 
  residuals. The root cause is the **overlapping returns problem** — consecutive 
  forward returns share 20 out of 21 days, making them almost identical by construction. 
  This renders standard errors unreliable and inflates t-statistics artificially.

- **Jarque-Bera test (p = 0.000)**: residuals are non-normal, with excess kurtosis of 
  4.33 and skew of -0.41 — fat tails remain despite winsorization.

The β estimates are still unbiased, but standard errors and p-values cannot be trusted.

### Correction Strategy

To address the overlapping returns problem, we resample the panel to **one observation 
per ticker every 21 trading days** — matching the horizon of the forward return. 
This ensures consecutive observations are non-overlapping and approximately independent, 
bringing the Durbin-Watson statistic closer to 2 and making standard errors reliable.

The sample size drops from 1.26 million to approximately 60,000 observations — a real 
reduction, but the remaining observations are statistically independent and therefore 
genuinely informative.

In [79]:
# ── Resample panel to non-overlapping observations ────────────────────
# Take one observation per ticker every 21 trading days
# This eliminates the overlapping returns problem

panel_monthly = (
    panel_clean
    .groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

print(f"Panel daily shape:   {panel_clean.shape}")
print(f"Panel monthly shape: {panel_monthly.shape}")
print(f"Observations lost:   {len(panel_clean) - len(panel_monthly)}")

# ── Run OLS on non-overlapping panel ─────────────────────────────────
X_m = sm.add_constant(panel_monthly[features])
y_m = panel_monthly['forward_return']

model_monthly = sm.OLS(y_m, X_m).fit()
print(model_monthly.summary())

Panel daily shape:   (1271523, 11)
Panel monthly shape: (60863, 10)
Observations lost:   1210660
                            OLS Regression Results                            
Dep. Variable:         forward_return   R-squared:                       0.020
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     154.5
Date:                Wed, 03 Jun 2026   Prob (F-statistic):          6.66e-259
Time:                        13:04:48   Log-Likelihood:                 66588.
No. Observations:               60863   AIC:                        -1.332e+05
Df Residuals:                   60854   BIC:                        -1.331e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------

### OLS Results — Corrected Model

After resampling to non-overlapping 21-day observations, the Durbin-Watson statistic 
improves dramatically from **0.109 to 2.078** — indicating no meaningful serial 
autocorrelation. The R² remains stable at **2.2%**, confirming the signal was genuine 
and not an artefact of overlapping returns.

Key changes in coefficients:
- **Mkt_RF** flips sign from positive to negative — the daily panel coefficient was 
  distorted by autocorrelation
- **SMB** and **HML** increase substantially — their effect was previously underestimated
- **HML** recovers significance (p=0.000) in the monthly model — it was suppressed by autocorrelation in the daily panel
- **CPI_YoY** remains the strongest negative predictor — robust across both models

Non-normality of residuals persists (Kurtosis = 4.24, Skew = -0.38) — an inherent feature of 
financial return distributions that cannot be fully eliminated through winsorization alone.

In [80]:
panel_monthly.to_csv('data/panel_monthly.csv', index=False)
print("Panel monthly saved")

Panel monthly saved


## 10. Final Considerations

- **Survivorship bias**: 130 tickers (26% of the universe) could not be downloaded 
  from yfinance, reintroducing survivorship bias. A commercial database such as 
  CRSP would eliminate this issue.

- **Sample period**: 2010-2026 includes highly anomalous regimes (COVID-19, 
  aggressive Fed tightening) that may not be representative of future market conditions.

- **Non-normal residuals**: despite winsorization, residuals retain excess kurtosis. 
  A rank-based transformation or robust regression would be more appropriate.

- **Standard errors**: clustered standard errors by ticker would further improve 
  inference by accounting for cross-sectional correlation.

- **Model complexity**: a pooled OLS is a first attempt. Fixed effects, 
  Fama-MacBeth, or machine learning models would likely improve predictive power.